# VLM-FO1 Easy Demo

This notebook allows you to run the VLM-FO1 model easily. Just run the cells one by one!

In [ ]:
# @title 1. Setup Environment
# @markdown Run this cell to install the necessary libraries. This may take a few minutes.

print("Installing dependencies... (this may take 2-3 minutes)")
!pip install --quiet git+https://github.com/wildcraft958/VLM-FO1.git
!pip install --quiet torch torchvision transformers accelerate pillow numpy matplotlib requests

print("✓ Installation complete!")

In [ ]:
# @title 2. Load Model
# @markdown Run this cell to load the VLM-FO1 model. It will download the model weights automatically.

import torch
from vlm_fo1.model.builder import load_pretrained_model
from PIL import Image
import matplotlib.pyplot as plt
from vlm_fo1.mm_utils import prepare_inputs, extract_predictions_to_bboxes, draw_bboxes_and_save
from vlm_fo1.task_templates import OD_template
import requests
from io import BytesIO
import os

# Use the default model
model_path = "omlab/VLM-FO1_Qwen2.5-VL-3B-v01"

print(f"Loading model: {model_path}...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

try:
    tokenizer, model, image_processors = load_pretrained_model(
        model_path,
        device=device,
        load_8bit=False # Set to True if you run out of memory on T4 GPU
    )
    print("✓ Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Try restarting the runtime and running again.")

In [ ]:
# @title 3. Run Inference
# @markdown Enter an image URL and a text query, then run this cell.

image_url = "https://raw.githubusercontent.com/om-ai-lab/VLM-FO1/main/demo/demo_image.jpg" # @param {type:"string"}
query = "orange" # @param {type:"string"}

# 1. Load and Display Image
if image_url.startswith("http"):
    response = requests.get(image_url)
    img = Image.open(BytesIO(response.content)).convert("RGB")
else:
    # Assume local path
    if os.path.exists(image_url):
        img = Image.open(image_url).convert("RGB")
    else:
        print(f"Error: File not found at {image_url}")
        raise FileNotFoundError(f"File not found: {image_url}")

# Save to temp file for the model processor
temp_image_path = "temp_input.jpg"
img.save(temp_image_path)

plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis('off')
plt.title(f"Query: {query}")
plt.show()

# 2. Prepare Inputs
bbox_list = [] 
messages = [{
    "role": "user",
    "content": [
        {"type": "image_url", "image_url": {"url": temp_image_path}},
        {"type": "text", "text": OD_template.format(query)},
    ],
    "bbox_list": bbox_list,
}]

generation_kwargs = prepare_inputs(
    model_path,
    model,
    image_processors,
    tokenizer,
    messages,
    max_tokens=512,
    top_p=0.05,
    temperature=0.0,
    do_sample=False,
)

# 3. Run Model
print("Running inference...")
with torch.inference_mode():
    output_ids = model.generate(**generation_kwargs)

outputs = tokenizer.decode(
    output_ids[0, generation_kwargs['inputs'].shape[1]:],
    skip_special_tokens=True
).strip()

print(f"Output: {outputs}")

# 4. Visualize Result
predicted_bboxes = extract_predictions_to_bboxes(outputs, bbox_list)
draw_bboxes_and_save(img, predicted_bboxes, "result.jpg")

res_img = Image.open("result.jpg")
plt.figure(figsize=(10, 8))
plt.imshow(res_img)
plt.axis('off')
plt.title("Result")
plt.show()